# genulens — Python binding examples

This notebook shows how to call genulens directly from Python without invoking the CLI binary.

**Setup**: build the project first, then launch Jupyter with the build directory on `PYTHONPATH`:
```bash
cmake --build build
PYTHONPATH=build jupyter notebook examples/python_binding.ipynb
```
Or install the package (`pip install -e .`) if a `setup.py`/`pyproject.toml` is available.

In [ ]:
import sys, os
# examples/ の一つ上がプロジェクトルート → build/ を探す
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
sys.path.insert(0, os.path.join(project_root, 'build'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import genulens
print('genulens loaded OK')

## 1. 最小構成で実行する

`genulens.Config` でパラメータを指定し、`genulens.simulate()` を呼ぶ。  
結果は `SimulationResult` で、`.to_numpy()` で numpy 配列に変換できる。

In [ ]:
cfg = genulens.Config(
    l=1.0,       # 銀経 [deg]
    b=-3.9,      # 銀緯 [deg]
    n_simu=100_000,
    seed=42,
)

result = genulens.simulate(cfg)
print('columns :', result.columns)
print('n_events:', result.to_numpy().shape[0])

In [ ]:
# numpy → pandas DataFrame
df = pd.DataFrame(result.to_numpy(), columns=result.columns)
df.head()

## 2. 重み付き tE 分布

`weight` 列は各モンテカルロサンプルのイベントレートの重み。  
統計量・ヒストグラムは必ず重み付きで扱う。

In [ ]:
w  = df['weight'].to_numpy()
tE = df['tE'].to_numpy()

# 重み付き中央値
order  = np.argsort(tE)
cumw   = np.cumsum(w[order]) / w.sum()
tE_med = tE[order][np.searchsorted(cumw, 0.5)]
print(f'weighted median tE = {tE_med:.1f} days')

bins = np.logspace(-1, 3, 60)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(tE, bins=bins, weights=w, density=True)
ax.axvline(tE_med, color='r', ls='--', label=f'median {tE_med:.0f} d')
ax.set_xscale('log')
ax.set_xlabel(r'$t_E$ [days]')
ax.set_ylabel('weighted probability density')
ax.set_title(f'(l, b) = ({cfg.l}, {cfg.b}) deg')
ax.legend()
plt.tight_layout()
plt.show()

## 3. レンズ成分ごとの寄与

成分番号: 0–6 薄円盤、7 厚円盤、8 バー、9 NSD、10 恒星ハロー

In [ ]:
comp_names = {
    0:'disk0', 1:'disk1', 2:'disk2', 3:'disk3',
    4:'disk4', 5:'disk5', 6:'disk6',
    7:'thick disk', 8:'bar', 9:'NSD', 10:'halo',
}

W_total = w.sum()
for comp_id, name in comp_names.items():
    mask = df['lens_component'] == comp_id
    frac = df.loc[mask, 'weight'].sum() / W_total
    if frac > 0:
        print(f'{name:12s}  {frac:.4f}')

## 4. 観測 tE を使った posterior distribution

`observed_tE` と `observed_tE_error` を設定すると、ガウス尤度で重み付けされる。

In [ ]:
cfg_fit = genulens.Config(l=1.0, b=-3.9, n_simu=100_000, seed=42)
cfg_fit.observed_tE       = 54.5   # 観測値 [days]
cfg_fit.observed_tE_error = 5.0    # 誤差 [days]

result_fit = genulens.simulate(cfg_fit)
df_fit = pd.DataFrame(result_fit.to_numpy(), columns=result_fit.columns)

w_fit  = df_fit['weight'].to_numpy()
tE_fit = df_fit['tE'].to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(tE_fit, bins=bins, weights=w_fit, density=True, label='posterior')
ax.axvline(cfg_fit.observed_tE, color='r', ls='--',
           label=f'$t_{{E,obs}}$ = {cfg_fit.observed_tE} d')
ax.set_xscale('log')
ax.set_xlabel(r'$t_E$ [days]')
ax.set_ylabel('weighted probability density')
ax.legend()
plt.tight_layout()
plt.show()

## 5. カスタム尤度関数

`likelihood` 引数に Python 関数を渡すと、内部の重みにその値が乗算される。  
関数は `Event` オブジェクトを受け取り `float` を返す。  
利用可能なフィールド: `weight`, `tE`, `thetaE`, `piE`,  
`lens_distance_pc`, `source_distance_pc`, `lens_mass_msun`,  
`mu_rel_masyr`, `lens_component`, `source_component`

In [ ]:
import math

tE_obs, tE_err     = 54.5, 5.0   # days
thetaE_obs, tE_err2 = 0.5, 0.05  # mas

def joint_like(event):
    x_tE     = (event.tE     - tE_obs)     / tE_err
    x_thetaE = (event.thetaE - thetaE_obs) / tE_err2
    return math.exp(-0.5 * (x_tE**2 + x_thetaE**2))

result_joint = genulens.simulate(cfg, likelihood=joint_like)
df_j = pd.DataFrame(result_joint.to_numpy(), columns=result_joint.columns)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, xlabel in zip(axes,
                            ['tE', 'lens_mass_msun'],
                            [r'$t_E$ [days]', r'$M_L$ [$M_\odot$]']):
    vals = df_j[col].to_numpy()
    wj   = df_j['weight'].to_numpy()
    b_   = np.logspace(np.log10(vals[vals>0].min()), np.log10(vals.max()), 50)
    ax.hist(vals, bins=b_, weights=wj, density=True)
    ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('weighted density')
plt.suptitle('joint tE + thetaE likelihood')
plt.tight_layout()
plt.show()

## 6. IMF パラメータを変えて比較

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for alpha2, label in [(-1.13, 'Kroupa (default)'), (-1.35, 'Salpeter-like')]:
    c = genulens.Config(l=1.0, b=-3.9, n_simu=50_000, seed=42)
    c.model.imf.alpha2 = alpha2
    r = genulens.simulate(c)
    d = pd.DataFrame(r.to_numpy(), columns=r.columns)
    ax.hist(d['tE'], bins=bins, weights=d['weight'],
            density=True, histtype='step', label=f'alpha2={alpha2} ({label})')

ax.set_xscale('log')
ax.set_xlabel(r'$t_E$ [days]')
ax.set_ylabel('weighted probability density')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. `ruc` ショートハンド

`genulens.ruc(l, b, n_simu, seed)` は `Config` を作らずに1行で実行できる簡易関数。

In [ ]:
r = genulens.ruc(l=0.0, b=-2.0, n_simu=10_000, seed=1)
d = pd.DataFrame(r.to_numpy(), columns=r.columns)
print(d[['tE', 'thetaE', 'lens_mass_msun', 'mu_rel_masyr']].describe())